# Metadata and Cold-Start Recommendation

Notebook 10 compared the models using interactions only. This notebook asks whether
product metadata - titles, descriptions and categories, encoded in Notebook 07 - adds
anything on top of that.

Only LightGCN is carried forward. NGCF and GraphSAGE both lost to matrix factorisation
in the collaborative setting, so adding features to them would be studying models that
are already behind. GraphSAGE appears once, in the cold-start experiment, because that
is the reason it was included in the first place: it is the one model that builds an item
embedding by aggregating features rather than by looking up an ID.

Three questions:

1. **Does metadata help?** LightGCN with BLaIR features against LightGCN without, on both
   protocols. The feature projection is initialised at zero, so the two models start
   numerically identical and any difference is attributable to the metadata alone.
2. **Does the encoder matter?** BLaIR against MiniLM and TF-IDF+SVD. BLaIR was
   pre-trained by the dataset authors on Amazon metadata, so the question is whether that
   domain-specific pre-training earns anything over generic alternatives.
3. **Does metadata help cold items?** Items that never appear in training have no useful
   ID embedding, only a description. If metadata helps anywhere, it should help here.

Everything else is inherited from Notebook 10 unchanged: same subset, splits, graphs,
evaluators, training loop and selected hyperparameters.

## 1. Imports and configuration

In [1]:
%load_ext autoreload
%autoreload 2

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import evaluation as ev
from src import features as ft
from src import graph as gr
from src import models as md
from src import training as tr

# Paths
SUBSETS_DIR = PROJECT_ROOT / "data" / "processed" / "subsets_earliest"
FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "features_earliest"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
CACHE = RESULTS_DIR / "11_runs"

for directory in (RESULTS_DIR, FIGURES_DIR, CACHE):
    directory.mkdir(parents=True, exist_ok=True)

# Experiment config, identical to Notebook 10
SUBSET = "small"
PROTOCOLS = ["last_out", "timestamp"]
K_VALUES = [10, 20]
EVAL_BATCH_USERS = 1024

SEEDS = [42, 1, 2]
SEED = SEEDS[0]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DIM = 32
LR = 1e-3
BATCH = 8192
SAMPLING = "user"
EPOCHS = 300

ENCODERS = ["blair", "minilm", "tfidf_svd"]

# selected in Notebook 10, carried forward unchanged
LIGHTGCN_BEST = {
    "last_out":  {"n_layers": 3, "l2_reg": 3e-3},
    "timestamp": {"n_layers": 3, "l2_reg": 3e-4},
}

# Notebook 10 test results, for comparison
NO_FEATURES = {
    "last_out":  {"ndcg@10": 0.00812, "std": 0.00007},
    "timestamp": {"ndcg@10": 0.00158, "std": 0.00020},
}

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"device: {DEVICE}")
print(f"subset: {SUBSET}, protocols: {PROTOCOLS}, seeds: {SEEDS}")
print(f"encoders: {ENCODERS}")

device: cuda
subset: small, protocols: ['last_out', 'timestamp'], seeds: [42, 1, 2]
encoders: ['blair', 'minilm', 'tfidf_svd']


## 2. Data, graphs, evaluators and features

The first four cells are Notebook 10's setup unchanged (same splits, same graphs, same
evaluators) so that the feature and no-feature arms are comparable without
qualification.

The new part is the feature matrices. Notebook 07 built these for every item in the
subset, including items that never appear in training, which is what makes the cold-start
experiment possible: a cold item has no useful ID embedding but it does have a
description.

In [3]:
def load_protocol(subset, protocol):
    base = SUBSETS_DIR / subset
    splits = base / "splits" / protocol
    parts = {
        name: pq.read_table(splits / f"{name}.parquet",
                            columns=["user_idx", "item_idx"]).to_pandas()
        for name in ["train", "validation", "test"]
    }
    return {
        "protocol": protocol,
        "n_users": len(pq.read_table(base / "user_mapping.parquet")),
        "n_items": len(pq.read_table(base / "item_mapping.parquet")),
        "eval_users": pq.read_table(
            splits / "eval_users.parquet").to_pandas()["user_idx"].to_numpy(),
        "cold_items": pq.read_table(
            splits / "cold_items.parquet").to_pandas()["item_idx"].to_numpy(),
        **parts,
    }


data = {p: load_protocol(SUBSET, p) for p in PROTOCOLS}

pd.DataFrame([{
    "protocol": p,
    "users": d["n_users"], "items": d["n_items"],
    "train": len(d["train"]), "validation": len(d["validation"]), "test": len(d["test"]),
    "eval_users": len(d["eval_users"]), "cold_items": len(d["cold_items"]),
} for p, d in data.items()])

,protocol,users,items,train,validation,test,eval_users,cold_items
0,last_out,258597,105298,1829582,258597,258597,50000,425
1,timestamp,258597,105298,2020041,165504,161231,50000,8353


In [4]:
graphs = {}
for protocol, d in data.items():
    n_nodes = d["n_users"] + d["n_items"]
    edge_index = gr.build_edge_index(d["train"], d["n_users"], d["n_items"])

    symmetric = gr.normalised_adjacency(edge_index, n_nodes, device=DEVICE)
    radius, iterations = gr.spectral_radius(symmetric)

    leaked = (gr.count_held_out_edges(edge_index, d["validation"], d["n_users"])
              + gr.count_held_out_edges(edge_index, d["test"], d["n_users"]))

    graphs[protocol] = {
        "edge_index": edge_index,
        "n_nodes": n_nodes,
        "symmetric": symmetric.to_sparse_csr(),
        "mean": gr.mean_adjacency(edge_index, n_nodes, device=DEVICE).to_sparse_csr(),
    }

    print(f"{protocol}: {edge_index.shape[1] // 2:,} edges over {n_nodes:,} nodes | "
          f"symmetric {gr.verify_symmetry(edge_index)} | "
          f"spectral radius {radius:.6f} ({iterations} it) | held-out edges {leaked}")

    assert leaked == 0
    assert abs(radius - 1.0) < 1e-3

c:\Projects\gnn-recommender-dissertation\src\graph.py:78: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  adj = torch.sparse_coo_tensor(
C:\Users\xhani\AppData\Local\Temp\ipykernel_7420\1024560151.py:15: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:51.)
  "symmetric": symmetric.to_sparse_csr(),


last_out: 1,829,582 edges over 363,895 nodes | symmetric True | spectral radius 1.000000 (43 it) | held-out edges 0
timestamp: 2,020,041 edges over 363,895 nodes | symmetric True | spectral radius 1.000000 (31 it) | held-out edges 0


In [5]:
test_evaluators, val_evaluators, tune_evaluators = {}, {}, {}

for protocol, d in data.items():
    test_evaluators[protocol] = ev.Evaluator.from_frames(
        ground_truth=d["test"],
        mask_frames=[d["train"], d["validation"]],
        eval_users=d["eval_users"],
        n_users=d["n_users"], n_items=d["n_items"],
        k_values=K_VALUES, device=DEVICE, batch_users=EVAL_BATCH_USERS,
    )

    probe = ev.Evaluator.from_frames(
        ground_truth=d["validation"], mask_frames=[d["train"]],
        eval_users=d["eval_users"], n_users=d["n_users"], n_items=d["n_items"],
        k_values=K_VALUES, device=DEVICE, batch_users=EVAL_BATCH_USERS,
    )
    usable = d["eval_users"][np.diff(probe.truth_indptr)[d["eval_users"]] > 0]

    for store, users in [(val_evaluators, usable), (tune_evaluators, usable[:10_000])]:
        store[protocol] = ev.Evaluator.from_frames(
            ground_truth=d["validation"], mask_frames=[d["train"]],
            eval_users=users, n_users=d["n_users"], n_items=d["n_items"],
            k_values=K_VALUES, device=DEVICE, batch_users=EVAL_BATCH_USERS,
        )

    print(f"{protocol}: test {len(d['eval_users']):,} users | "
          f"validation {len(usable):,} | tuning {len(tune_evaluators[protocol].eval_users):,}")

last_out: test 50,000 users | validation 50,000 | tuning 10,000
timestamp: test 50,000 users | validation 24,514 | tuning 10,000


In [6]:
features = {enc: ft.load_item_features(FEATURES_DIR, SUBSETS_DIR, SUBSET, encoder=enc)
            for enc in ENCODERS}

n_items = data["last_out"]["n_items"]
rows = []
for enc, matrix in features.items():
    assert matrix.shape[0] == n_items, f"{enc}: {matrix.shape[0]} rows, expected {n_items}"
    rows.append({"encoder": enc, "dimensions": matrix.shape[1],
                 "mean": float(matrix.mean()), "std": float(matrix.std()),
                 "all-zero rows": int((np.abs(matrix).sum(axis=1) == 0).sum())})

pd.DataFrame(rows)

,encoder,dimensions,mean,std,all-zero rows
0,blair,807,0.001247,0.049767,0
1,minilm,423,0.002407,0.068719,0
2,tfidf_svd,295,0.003276,0.082274,0


In [7]:
def build_lightgcn(protocol, encoder=None, seed=SEED, n_layers=None):
    """LightGCN at the Notebook 10 configuration, optionally with item features."""
    d, cfg = data[protocol], LIGHTGCN_BEST[protocol]
    return md.LightGCN(
        d["n_users"], d["n_items"], dim=DIM, seed=seed,
        item_features=None if encoder is None else features[encoder],
        adjacency=graphs[protocol]["symmetric"],
        n_layers=cfg["n_layers"] if n_layers is None else n_layers,
    )

## 3. Regularisation with features

The feature projection adds 25,824 parameters to a model that already has 11.6 million,
so the best regularisation probably does not move. But "probably" is not a reason to skip
the check, and if it has moved then every result below would be comparing a tuned
feature-free model against an untuned feature model.

Three values around the Notebook 10 optimum, on leave-last-out, one seed. The no-feature
result at the same setting comes from Notebook 10's cache, so the comparison is direct.

In [8]:
CACHE_10 = RESULTS_DIR / "10_tuning"


def tune_run(protocol, encoder, l2_reg, seed=SEED, epochs=EPOCHS):
    """Train against the tuning evaluator, cached to disk."""
    enc = encoder or "none"
    tag = f"tune_{protocol}_{enc}_reg{l2_reg:g}_e{epochs}_s{seed}"
    path = CACHE / f"{tag}.csv"

    if path.exists():
        history = pd.read_csv(path)
    else:
        t0 = time.time()
        outcome = tr.train_bpr(
            build_lightgcn(protocol, encoder, seed),
            data[protocol]["train"], data[protocol]["n_items"],
            evaluator=tune_evaluators[protocol],
            epochs=epochs, batch_size=BATCH, learning_rate=LR, l2_reg=l2_reg,
            sampling=SAMPLING, patience=10, eval_every=10,
            selection_metric="ndcg@10", device=DEVICE, seed=seed, verbose=False,
        )
        history = outcome["history"]
        history.to_csv(path, index=False)
        print(f"  {tag}: {(time.time() - t0) / 60:.1f} min", flush=True)

    evaluated = history.dropna(subset=["val_ndcg@10"])
    best = evaluated.loc[evaluated["val_ndcg@10"].idxmax()]
    return {"protocol": protocol, "encoder": enc, "l2_reg": l2_reg, "seed": seed,
            "best_epoch": int(best["epoch"]),
            "val_ndcg@10": float(best["val_ndcg@10"]),
            "epochs_run": len(history)}


def notebook10_run(protocol, l2_reg, n_layers=3, seed=SEED):
    """Read the matching feature-free run from Notebook 10's cache."""
    path = CACHE_10 / f"LightGCN_{protocol}_L{n_layers}_reg{l2_reg:g}_e300_s{seed}.csv"
    evaluated = pd.read_csv(path).dropna(subset=["val_ndcg@10"])
    best = evaluated.loc[evaluated["val_ndcg@10"].idxmax()]
    return {"protocol": protocol, "encoder": "none", "l2_reg": l2_reg, "seed": seed,
            "best_epoch": int(best["epoch"]),
            "val_ndcg@10": float(best["val_ndcg@10"]),
            "epochs_run": len(pd.read_csv(path))}

In [9]:
reg_check = pd.DataFrame(
    [notebook10_run("last_out", reg) for reg in [1e-3, 3e-3, 1e-2]]
    + [tune_run("last_out", "blair", reg) for reg in [1e-3, 3e-3, 1e-2]]
)
reg_check.pivot(index="l2_reg", columns="encoder", values="val_ndcg@10")

c:\Projects\gnn-recommender-dissertation\src\training.py:65: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  users = torch.as_tensor(train["user_idx"].to_numpy(), dtype=torch.long, device=device)
c:\Projects\gnn-recommender-dissertation\src\training.py:128: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  epoch_loss += float(loss) * len(batch)


  tune_last_out_blair_reg0.001_e300_s42: 37.4 min
  tune_last_out_blair_reg0.003_e300_s42: 36.8 min
  tune_last_out_blair_reg0.01_e300_s42: 37.7 min


encoder,blair,none
l2_reg,,
0.001,0.013894,0.012407
0.003,0.012991,0.012740
0.010,0.011956,0.011910


In [10]:
reg_check = pd.DataFrame(
    [notebook10_run("last_out", reg) for reg in [1e-3, 3e-3, 1e-2]]
    + [tune_run("last_out", "blair", reg)
       for reg in [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]]
)
reg_check.pivot(index="l2_reg", columns="encoder", values="val_ndcg@10")

  tune_last_out_blair_reg0.0001_e300_s42: 37.1 min
  tune_last_out_blair_reg0.0003_e300_s42: 36.9 min


encoder,blair,none
l2_reg,,
0.0001,0.014711,NaN
0.0003,0.014607,NaN
0.0010,0.013894,0.012407
0.0030,0.012991,0.012740
0.0100,0.011956,0.011910


In [11]:
reg_check[reg_check.encoder == "blair"][
    ["l2_reg", "best_epoch", "val_ndcg@10", "epochs_run"]]

,l2_reg,best_epoch,val_ndcg@10,epochs_run
3,0.0001,220,0.014711,300
4,0.0003,220,0.014607,300
5,0.0010,300,0.013894,300
6,0.0030,300,0.012991,300
7,0.0100,290,0.011956,300


In [12]:
reg_check_ts = pd.DataFrame(
    [notebook10_run("timestamp", reg) for reg in [3e-4, 1e-3]]
    + [tune_run("timestamp", "blair", reg) for reg in [3e-5, 1e-4, 3e-4]]
)
reg_check_ts.pivot(index="l2_reg", columns="encoder", values="val_ndcg@10")

  tune_timestamp_blair_reg3e-05_e300_s42: 43.5 min
  tune_timestamp_blair_reg0.0001_e300_s42: 43.6 min
  tune_timestamp_blair_reg0.0003_e300_s42: 43.7 min


encoder,blair,none
l2_reg,,
0.00003,0.006142,NaN
0.00010,0.006027,NaN
0.00030,0.005868,0.005309
0.00100,NaN,0.005178


## 4. Does metadata help?

LightGCN with BLaIR features against LightGCN without, on both protocols, three seeds,
evaluated once on test. The feature-free arm is not retrained --> it comes from Notebook
10's saved results, which used the same splits, evaluators, training loop and seeds.

Each arm uses its own regularisation: 3e-3 and 3e-4 without features, 1e-4 and 3e-5 with
them. Using a single value for both would have understated the effect several-fold, as
the tuning section shows.

Per-user scores are saved so Notebook 12 can test whether the difference is significant
user by user rather than only on the means.

In [13]:
FINAL_CACHE = RESULTS_DIR / "11_final"
FINAL_CACHE.mkdir(exist_ok=True)
FINAL_CACHE_10 = RESULTS_DIR / "10_final"

FEATURE_REG = {"last_out": 1e-4, "timestamp": 3e-5}


def final_run(protocol, encoder, l2_reg, seed):
    """Train at the selected configuration and evaluate once on test."""
    enc = encoder or "none"
    tag = f"LightGCN_{enc}_{protocol}_s{seed}"
    summary_path = FINAL_CACHE / f"{tag}.json"

    if summary_path.exists():
        with open(summary_path) as fh:
            return {**json.load(fh), "cached": True}

    t0 = time.time()
    outcome = tr.train_bpr(
        build_lightgcn(protocol, encoder, seed),
        data[protocol]["train"], data[protocol]["n_items"],
        evaluator=val_evaluators[protocol],
        epochs=EPOCHS, batch_size=BATCH, learning_rate=LR, l2_reg=l2_reg,
        sampling=SAMPLING, patience=10, eval_every=10, selection_metric="ndcg@10",
        device=DEVICE, seed=seed, verbose=False,
    )

    scored = test_evaluators[protocol].evaluate(outcome["model"].score_fn(),
                                                return_per_user=True)
    np.savez_compressed(FINAL_CACHE / f"{tag}.npz", **scored["per_user"])

    summary = {"model": "LightGCN", "encoder": enc, "protocol": protocol, "seed": seed,
               "l2_reg": l2_reg, "best_epoch": outcome["best_epoch"],
               "val_ndcg@10": float(outcome["best_val_score"]),
               "minutes": (time.time() - t0) / 60, **scored["means"]}
    with open(summary_path, "w") as fh:
        json.dump(summary, fh)

    print(f"  {tag}: {summary['minutes']:.1f} min, test ndcg@10 {summary['ndcg@10']:.5f}",
          flush=True)
    return {**summary, "cached": False}


def notebook10_final(protocol, seed):
    """The feature-free arm, from Notebook 10's saved results."""
    with open(FINAL_CACHE_10 / f"LightGCN_{protocol}_s{seed}.json") as fh:
        summary = json.load(fh)
    return {**summary, "encoder": "none",
            "l2_reg": LIGHTGCN_BEST[protocol]["l2_reg"], "cached": True}

In [14]:
main_results = pd.DataFrame(
    [notebook10_final(p, s) for p in PROTOCOLS for s in SEEDS]
    + [final_run(p, "blair", FEATURE_REG[p], s) for p in PROTOCOLS for s in SEEDS]
)
main_results.to_csv(RESULTS_DIR / "11_main_comparison.csv", index=False)

summary = (main_results.groupby(["protocol", "encoder"])["ndcg@10"]
           .agg(["mean", "std"]).round(5))
print(summary.to_string())

for protocol in PROTOCOLS:
    with_f = main_results.query("protocol == @protocol and encoder == 'blair'")["ndcg@10"].mean()
    without = main_results.query("protocol == @protocol and encoder == 'none'")["ndcg@10"].mean()
    print(f"\n{protocol}: {without:.5f} -> {with_f:.5f}  ({with_f / without - 1:+.1%})")

  LightGCN_blair_last_out_s42: 25.5 min, test ndcg@10 0.00872
  LightGCN_blair_last_out_s1: 22.5 min, test ndcg@10 0.00851
  LightGCN_blair_last_out_s2: 29.9 min, test ndcg@10 0.00868
  LightGCN_blair_timestamp_s42: 45.4 min, test ndcg@10 0.00237
  LightGCN_blair_timestamp_s1: 44.6 min, test ndcg@10 0.00229
  LightGCN_blair_timestamp_s2: 37.1 min, test ndcg@10 0.00224
                      mean      std
protocol  encoder                  
last_out  blair    0.00864  0.00011
          none     0.00812  0.00007
timestamp blair    0.00230  0.00007
          none     0.00158  0.00020

last_out: 0.00812 -> 0.00864  (+6.3%)

timestamp: 0.00158 -> 0.00230  (+45.9%)
